# Detecting Fake News in Spanish with Classical NLP

This project explores whether simple NLP techniques can detect patterns associated with misinformation in Spanish news articles.

Using the Spanish Fake News Corpus (FakeDeS), we train and evaluate classical machine learning models based on TF-IDF features.

The goal is not to "detect truth", but to analyze whether stylistic and linguistic patterns associated with misinformation can be captured by simple models.

## Dataset

This project uses the **Spanish Fake News Corpus Version 2.0 (FakeDeS Task @ Iberlef 2021)**, developed for the IberLEF shared task on fake news detection in Spanish.

The dataset contains pairs of real and fake news articles collected from online media and fact-checking websites.

News articles were labeled based on verification from established media outlets and specialized fact-checking platforms.

Topics include:

- Politics
- Society
- Science
- Covid-19
- Environment
- International
- Sport

Each entry in the dataset contains the following fields:

- **Id** – unique identifier for the news article  
- **Category** – label indicating whether the article is *True* or *Fake*  
- **Topic** – thematic category of the news  
- **Source** – media outlet or platform where the article appeared  
- **Headline** – title of the news article  
- **Text** – full news article content  
- **Link** – URL of the original source

The corpus is balanced between **true and fake news**.

Dataset splits:

- Train set
- Development set
- Test set

For the final model training, the **train and development sets are combined**, while the **test set is used only for final evaluation**.

https://huggingface.co/datasets/sayalaruano/FakeNewsCorpusSpanish

### Citation

If you use this dataset, please cite:

1. Gómez-Adorno, H., Posadas-Durán, J. P., Enguix, G. B., & Capetillo, C. P. (2021). Overview of FakeDeS at IberLEF 2021: Fake News Detection in Spanish Shared Task. Procesamiento del Lenguaje Natural, 67, 223-231.

2. Aragón, M. E., Jarquín, H., Gómez, M. M. Y., Escalante, H. J., Villaseñor-Pineda, L., Gómez-Adorno, H., ... & Posadas-Durán, J. P. (2020, September). Overview of mex-a3t at iberlef 2020: Fake news and aggressiveness analysis in mexican spanish. In Notebook Papers of 2nd SEPLN Workshop on Iberian Languages Evaluation Forum (IberLEF), Malaga, Spain.

3. Posadas-Durán, J. P., Gómez-Adorno, H., Sidorov, G., & Escobar, J. J. M. (2019). Detection of fake news in a new corpus for the Spanish language. Journal of Intelligent & Fuzzy Systems, 36(5), 4869-4876.

Dataset license: CC-BY-4.0

## Environment Setup

We begin by importing the required Python libraries for data processing, natural language processing, and machine learning.

In [ ]:
# Libraries
import pandas as pd
import numpy as np

# Machine learning
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix

# Stopwords
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
spanish_stopwords = stopwords.words("spanish")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## Loading the Dataset

In [ ]:
train_df = pd.read_csv("train.csv")
dev_df = pd.read_csv("development.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
print("Train shape:", train_df.shape)
print("Development shape:", dev_df.shape)
print("Test shape:", test_df.shape)

Train shape: (676, 7)
Development shape: (295, 7)
Test shape: (572, 7)


In [ ]:
# Normalize column names to avoid inconsistencies

train_df.columns = train_df.columns.str.lower()
dev_df.columns = dev_df.columns.str.lower()
test_df.columns = test_df.columns.str.lower()

### Combining Train and Development Sets

The train and development splits are combined to maximize the amount of training data.  
The test set is kept separate for final evaluation.

In [ ]:
# Combine train and development sets for training

full_train = pd.concat([train_df, dev_df], ignore_index=True)

print("Training set size:", full_train.shape)
print("Test set size:", test_df.shape)

Training set size: (971, 7)
Test set size: (572, 7)


## Data Preprocessing

In this step, we prepare the textual data for modeling.  
We combine the news headline and article text into a single field, handle missing values, and convert labels into a numerical format suitable for machine learning models.

In [ ]:
# Combine headline and article text
full_train["full_text"] = (
    full_train["headline"].fillna("") + " " + full_train["text"].fillna("")
)

test_df["full_text"] = (
    test_df["headline"].fillna("") + " " + test_df["text"].fillna("")
)

In [ ]:
# Convert labels to numeric format (Fake=0, True=1)
label_map = {"Fake": 0, "True": 1}

full_train["category"] = full_train["category"].map(label_map)

In [ ]:
# Define training and test variables
X_train = full_train["full_text"]
y_train = full_train["category"]

X_test = test_df["full_text"]
y_test = test_df["category"]

In [ ]:
print("Training examples:", X_train.shape)
print("Test examples:", X_test.shape)

Training examples: (971,)
Test examples: (572,)


## Baseline Model: Logistic Regression

As a baseline approach, we train a classical NLP model using **TF-IDF features** combined with **Logistic Regression**.

This type of model is widely used in text classification tasks because it is simple, interpretable, and often performs well on small datasets.

In [ ]:
# Baseline model: TF-IDF + Logistic Regression
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words=spanish_stopwords,
        max_features=5000
    )),
    ("clf", LogisticRegression(max_iter=1000))
])

In [ ]:
# Train the model
pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000,
                                 stop_words=['de', 'la', 'que', 'el', 'en', 'y',
                                             'a', 'los', 'del', 'se', 'las',
                                             'por', 'un', 'para', 'con', 'no',
                                             'una', 'su', 'al', 'lo', 'como',
                                             'más', 'pero', 'sus', 'le', 'ya',
                                             'o', 'este', 'sí', 'porque', ...])),
                ('clf', LogisticRegression(max_iter=1000))])

### Model Evaluation

In [ ]:
# Generate predictions
y_pred = pipeline.predict(X_test)

# Evaluation metrics
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.75      0.72       286
           1       0.72      0.65      0.69       286

    accuracy                           0.70       572
   macro avg       0.70      0.70      0.70       572
weighted avg       0.70      0.70      0.70       572

[[215  71]
 [ 99 187]]


## Model Comparison: Linear SVM

To evaluate whether model performance depends on the choice of algorithm, we train a second classifier using **Linear Support Vector Machines (SVM)**.

Linear SVMs are commonly used in text classification tasks and often perform well with high-dimensional representations such as TF-IDF.

In [ ]:
# TF-IDF + Linear SVM
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words=spanish_stopwords,
        max_features=5000
    )),
    ("clf", LinearSVC())
])

In [ ]:
# Train SVM model
svm_pipeline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000,
                                 stop_words=['de', 'la', 'que', 'el', 'en', 'y',
                                             'a', 'los', 'del', 'se', 'las',
                                             'por', 'un', 'para', 'con', 'no',
                                             'una', 'su', 'al', 'lo', 'como',
                                             'más', 'pero', 'sus', 'le', 'ya',
                                             'o', 'este', 'sí', 'porque', ...])),
                ('clf', LinearSVC())])

In [ ]:
# Predictions
y_pred_svm = svm_pipeline.predict(X_test)

# Evaluation
print(classification_report(y_test, y_pred_svm))
print(confusion_matrix(y_test, y_pred_svm))

              precision    recall  f1-score   support

           0       0.72      0.60      0.65       286
           1       0.66      0.76      0.71       286

    accuracy                           0.68       572
   macro avg       0.69      0.68      0.68       572
weighted avg       0.69      0.68      0.68       572

[[172 114]
 [ 68 218]]


## Model Interpretation

One advantage of linear models such as Logistic Regression is that they are interpretable.  
Each feature (word) receives a weight indicating how strongly it contributes to predicting a class.

By analyzing these coefficients, we can identify which terms are most strongly associated with **fake** or **true** news in the dataset.

In [ ]:
# Extract vectorizer and trained model
vectorizer = pipeline.named_steps["tfidf"]
model = pipeline.named_steps["clf"]

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

In [ ]:
import numpy as np

top_true_indices = np.argsort(coefficients)[-20:]
top_fake_indices = np.argsort(coefficients)[:20]

top_true_words = feature_names[top_true_indices]
top_fake_words = feature_names[top_fake_indices]

print("Words most associated with TRUE news:")
print(top_true_words)

print("\nWords most associated with FAKE news:")
print(top_fake_words)

Words most associated with TRUE news:
['instagram' 'investigación' 'julio' 'desarrollo' 'proceso' 'francisco'
 'tres' 'ciento' 'cuatro' 'coalición' 'mensaje' 'pasado' 'destacó'
 'domingo' 'cinco' 'aseguró' 'electoral' 'través' 'lópez' 'number']

Words most associated with FAKE news:
['amlo' 'pues' 'nombre' 'así' 'solo' 'usd' 'si' 'lоs' 'poder' 'nadie'
 'mencionó' 'descubren' 'persona' 'puntualizó' 'verdad' 'mexicanos'
 'declaraciones' 'selección' 'podrá' 'señalando']


## Error Analysis

To better understand the limitations of the model, we examine examples where the classifier produced incorrect predictions.

Analyzing these cases helps identify patterns that are difficult for simple NLP models to capture.

In [ ]:
results_df = pd.DataFrame({
    "headline": test_df["headline"],
    "true_label": y_test,
    "pred_label": y_pred
})

errors = results_df[results_df["true_label"] != results_df["pred_label"]]

errors.head(10)

,headline,true_label,pred_label
1,El Gobierno podrá acceder a las IPs de los móv...,0,1
8,Francia decreta el estado de emergencia y el t...,0,1
10,Despiden al gerente de Mets por acoso sexual a...,1,0
12,"¡AY, PAPÁ! Pablo Iglesias pierde juicio y es d...",0,1
18,La pandemia quizá esté reduciendo los nacimien...,1,0
25,Colau permite que ‘Tsunami’ anuncie en las mar...,0,1
26,“Prefiero un feminismo diverso que sea capaz d...,1,0
32,Desmanes y ataques contra 10 CAI dejan varios ...,1,0
37,El PSOE permite que el catalán y el árabe sean...,0,1
40,Este es el sorprendente hospital construido en...,0,1


## Limitations

While the models achieve moderate performance, several limitations should be considered.

First, the dataset is relatively small, which limits the ability of machine learning models to learn robust linguistic patterns.

Second, fake news detection cannot rely solely on textual features. Many false claims are written in a style similar to legitimate journalism, making them difficult to detect using simple NLP techniques.

Third, the model appears to rely on vocabulary patterns specific to the dataset. As topics and language evolve over time, models trained on historical data may not generalize well to new forms of misinformation.

Finally, automated systems should be seen as **tools to assist human fact-checkers**, rather than systems capable of determining truth on their own.

## Conclusion

This project explored whether simple NLP techniques could identify patterns associated with misinformation in Spanish news.

Using TF-IDF representations and classical machine learning models, we achieved around **70% accuracy** on the test dataset. While the models capture some stylistic signals associated with misinformation, error analysis shows that distinguishing fake news from legitimate journalism remains a challenging task.

These results highlight both the potential and the limitations of automated approaches to misinformation detection.
